# MLP-only baseline on foraging: PPO vs DDPG vs ES

The **"nothing given"** end of the steering ladder. Unlike the NCAP runs, this baseline
has *no circuit prior and no steering controller*: a plain fully-connected policy reads
the whole observation (joint angles, the egocentric `to_target` vector, body velocities)
and has to learn **both** swimming and steering to the food from scratch.

We train that same MLP baseline three ways and compare them on physics-only foraging
success:

| method | family | policy factory |
|---|---|---|
| `ppo`  | on-policy RL (tonic)   | `ppo_mlp_model` |
| `ddpg` | off-policy RL (tonic)  | `ddpg_mlp_model` |
| `es`   | Evolution Strategies   | `MLPSwimmerPolicy` |

**Reward.** `foraging` ships with every reward term at `0.0` (see `envs.foraging`), so a
run with no `task_kwargs` has *nothing to learn from*. We use the same shaping as the
`learned_steering` PoC — a dense **progress** reward (distance closed) plus an **eat
bonus**, swim-speed off — so these numbers sit on the same axis as that notebook's.

**What to expect.** This is a genuine baseline *measurement*, not a tuned result: the MLP
must discover the turn-to-food behaviour the NCAP `steer_to_food` reflex gets for free
(the from-random-init learned steerer only reached ~7% — see `STEERING.md`). Read the
success bars as "how far does an unstructured policy get on its own." Runs top-to-bottom;
training is skipped for any run already on disk.

In [ ]:
# --- Setup -------------------------------------------------------------------------
%matplotlib inline
import sys
from pathlib import Path

# This notebook only orchestrates; every function/class lives in macrocircuits under src/.
SRC = str(Path.cwd().parent / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import os
import numpy as np
import matplotlib.pyplot as plt

from macrocircuits import ensure_tonic
ensure_tonic()  # clone neuromatch/tonic next to this notebook (once) and put it on the path

import torch, tonic, tonic.torch
from dm_control.mujoco import engine
from IPython.display import Video

from macrocircuits.es import MLPSwimmerPolicy, es_config, is_es_trained, run_es
from macrocircuits.training import resolve_runs, run_config, run_path, is_trained, train
from macrocircuits.plotting import plot_performance
from macrocircuits.video import write_video

# --- Knobs -------------------------------------------------------------------------
N_JOINTS      = 5          # 6-link swimmer
SUCCESS_DIST  = 0.15       # head within this of the food during an episode => that episode found it
EVAL_SEED     = 0          # which fresh episodes success_rate draws; numbers shift ~+/-10% across seeds
EVAL_EPISODES = 30

# Training budgets -- the main knobs for a longer/shorter run.
STEPS_RL       = int(1e5)  # PPO / DDPG env steps (a couple of minutes each on CPU)
ES_GENERATIONS = 20        # ES generations; population 64 => the slow axis (see the Train section)

# foraging ships rewardless; give it the same dense signal as the learned_steering PoC,
# so a learned baseline actually has something to climb.
FORAGE_REWARD = dict(progress_reward_weight=10.0, eat_bonus=5.0)
print('ready')

## Helpers

Physics-only evaluation and rendering, shared by all three methods. The one wrinkle: a
tonic RL checkpoint (PPO/DDPG) and an ES checkpoint load differently — `trained_policy`
rebuilds a tonic agent from `config.yaml`; `es_policy` rebuilds a bare `MLPSwimmerPolicy`
state_dict — but both return the same `(policy, env, physics)` triple, so one
`success_rate` scores them all. Every eval env is the **no-time-feature** foraging env the
MLP baseline is trained on (`network='mlp'` sets `time_feature` off).

In [ ]:
def _get_physics(env):
    while env is not None:
        if hasattr(env, 'physics'):
            return env.physics
        inner = getattr(env, 'environment', None)
        if inner is not None and hasattr(inner, 'physics'):
            return inner.physics
        env = getattr(env, 'env', None)
    raise RuntimeError('no physics in env chain')


def _arena_frame(physics, distance=5.5):
    """Fixed top-down camera on the whole arena, so you see the worm traverse to food."""
    cam = engine.Camera(physics, height=480, width=640, camera_id=-1)
    rc = cam._render_camera
    rc.lookat[:] = [0.0, 0.0, 0.0]
    rc.distance = distance; rc.azimuth = 90; rc.elevation = -90
    frame = cam.render(); cam._scene.free()
    return frame


def _mlp_eval_env(seed=EVAL_SEED):
    """A distributed foraging env with NO time feature -- the observation the MLP baseline
    is trained and evaluated on. The reward shaping is irrelevant here: success_rate is
    physics-only, so the env reward is never read."""
    env = tonic.environments.distribute(
        lambda: tonic.environments.ControlSuite('swimmer-foraging'))
    env.initialize(seed)
    return env


def trained_policy(path, seed=EVAL_SEED):
    """Load a tonic RL checkpoint (PPO or DDPG) from `path`; returns (policy, env, physics).
    Rebuilds the exact agent/env from the run's config.yaml, then loads the last step_*."""
    import argparse, yaml
    import macrocircuits.training as _t
    ns = dict(vars(_t))
    cfg = argparse.Namespace(
        **yaml.load(open(os.path.join(path, 'config.yaml')), Loader=yaml.FullLoader))
    try:
        if cfg.header: exec(cfg.header, ns)
    except Exception:
        pass
    agent = eval(cfg.agent, ns)
    env = tonic.environments.distribute(lambda: eval(cfg.environment, ns))
    env.initialize(seed)
    agent.initialize(observation_space=env.observation_space,
                     action_space=env.action_space, seed=seed)
    cp = os.path.join(path, 'checkpoints')
    ids = [int(n.split('.')[0][5:]) for n in os.listdir(cp) if n.startswith('step_')]
    agent.load(os.path.join(cp, f'step_{max(ids)}'))
    return (lambda obs: agent.test_step(obs, 0)), env, _get_physics(env.environments[0])


def es_policy(path, seed=EVAL_SEED):
    """Load an ES-trained MLP policy from `path`; returns (policy, env, physics).
    An ES checkpoint is a bare MLPSwimmerPolicy state_dict (not a tonic agent), so it is
    rebuilt from config.yaml and wrapped to match the same eval env."""
    import yaml
    with open(os.path.join(path, 'config.yaml')) as f:
        cfg = yaml.safe_load(f)
    env = _mlp_eval_env(seed)
    policy = MLPSwimmerPolicy(env.observation_space.shape[-1], env.action_space.shape[-1],
                              hidden_sizes=tuple(cfg.get('hidden_sizes') or (64, 64)))
    policy.load_state_dict(
        torch.load(os.path.join(path, 'checkpoints', 'best.pt'), map_location='cpu'))
    policy.eval()
    def policy_fn(obs):
        with torch.no_grad():
            return policy(torch.as_tensor(obs, dtype=torch.float32)).numpy()
    return policy_fn, env, _get_physics(env.environments[0])


def untrained_mlp_policy(seed=EVAL_SEED):
    """A fresh, random-init MLP baseline -- the floor the trained runs have to beat."""
    torch.manual_seed(seed)
    env = _mlp_eval_env(seed)
    policy = MLPSwimmerPolicy(env.observation_space.shape[-1], env.action_space.shape[-1])
    policy.eval()
    def policy_fn(obs):
        with torch.no_grad():
            return policy(torch.as_tensor(obs, dtype=torch.float32)).numpy()
    return policy_fn, env, _get_physics(env.environments[0])


def load_policy(run, seed=EVAL_SEED):
    """Dispatch on method: ES checkpoints load differently from tonic RL ones."""
    path = run_path(**run)
    return es_policy(path, seed) if run['method'] == 'es' else trained_policy(path, seed)


def success_rate(policy, env, n_episodes=EVAL_EPISODES):
    """Fraction of fresh episodes whose head comes within SUCCESS_DIST of the food.
    Distance is read from infos['observations'] (the pre-reset transition obs), never
    from a just-reset env."""
    tgt = slice(N_JOINTS, N_JOINTS + 2)
    obs = env.start(); mind = np.inf; hits = []
    while len(hits) < n_episodes:
        obs, infos = env.step(policy(obs))
        mind = min(mind, float(np.linalg.norm(infos['observations'][0, tgt])))
        if infos['resets'][0]:
            hits.append(mind < SUCCESS_DIST); mind = np.inf
    return float(np.mean(hits))


def forage_episode(policy, env, physics, steps=1000, viz_food_size=0.10):
    """One episode; records overhead frames, head path, food positions, and the steps a
    food was reached (it then respawns). viz_food_size only enlarges the food marker for
    the video -- the success metric uses the true tiny food."""
    obs = env.start()
    if viz_food_size:
        physics.named.model.geom_size['target', 0] = viz_food_size; physics.forward()
    frames, hx, hy, fx, fy, eats = [], [], [], [], [], []
    prev_food = None
    for i in range(steps):
        frames.append(_arena_frame(physics))
        nose = physics.named.data.geom_xpos['nose'][:2].copy()
        food = physics.named.data.geom_xpos['target'][:2].copy()
        hx.append(nose[0]); hy.append(nose[1]); fx.append(food[0]); fy.append(food[1])
        if prev_food is not None and np.linalg.norm(food - prev_food) > 1e-6:
            eats.append(i)
        prev_food = food
        obs, infos = env.step(policy(obs))
        if infos['resets'][0]:
            break
    return dict(frames=frames, hx=np.array(hx), hy=np.array(hy),
                fx=np.array(fx), fy=np.array(fy), eats=eats)


def show_video(frames, path, fps=30):
    """Write frames to mp4 and embed directly (plays reliably in the VSCode notebook viewer)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    write_video(path, frames, fps=fps)
    return Video(path, embed=True, width=560)


def plot_path(ep, title):
    plt.figure(figsize=(6.5, 6.5))
    plt.scatter(ep['hx'], ep['hy'], c=np.arange(len(ep['hx'])), cmap='viridis', s=6, label='head path')
    food_pts = np.unique(np.c_[ep['fx'], ep['fy']], axis=0)
    plt.scatter(food_pts[:, 0], food_pts[:, 1], marker='*', s=260, c='red', ec='k', label='food', zorder=5)
    if ep['eats']:
        plt.scatter(ep['hx'][ep['eats']], ep['hy'][ep['eats']], s=90, c='lime', ec='k', label='reached', zorder=6)
    plt.gca().set_aspect('equal'); plt.grid(alpha=0.3); plt.legend()
    plt.title(title); plt.xlabel('world x'); plt.ylabel('world y'); plt.show()

print('helpers ready')

## The three runs

All are `network='mlp'` on `foraging` with the same reward shaping and seed — only the
training algorithm differs. Distinct `*_forage_poc` labels keep them off the directories
that `step1_swimmer.ipynb` uses for its own `mlp_ppo_foraging` / `mlp_ddpg_foraging` runs
(which train on the *unshaped* default reward), so the two notebooks never clobber each
other.

In [ ]:
DEFAULTS = dict(network='mlp', task='foraging', task_kwargs=FORAGE_REWARD, seed=EVAL_SEED)

RUNS = resolve_runs(
    [
        dict(method='ppo',  steps=STEPS_RL,             label='mlp_ppo_forage_poc'),
        dict(method='ddpg', steps=STEPS_RL,             label='mlp_ddpg_forage_poc'),
        dict(method='es',   generations=ES_GENERATIONS, label='mlp_es_forage_poc'),
    ],
    defaults=DEFAULTS,
)

for run in RUNS:
    budget = (f'{run["generations"]:>7,} generations' if run['method'] == 'es'
              else f'{run["steps"]:>7,} steps      ')
    print(f'{run["label"]:<22} {run["method"]:<5} {budget}  ->  {run_path(**run)}')

## Train

Trains each run, skipping any whose exact parameters are already checkpointed. Budgets are
the constants above.

> **Cost.** PPO and DDPG at `1e5` steps are ~2-3 min each on CPU. **ES is the slow axis**:
> it rolls out its whole population (64) every generation, so `ES_GENERATIONS=20` is
> ~1.3k episodes of true environment interaction — think ~10-30 min on CPU. Drop
> `ES_GENERATIONS` to ~5 for a quick smoke test, or raise it for a fairer comparison.

In [ ]:
import time
start = time.time()

for run in RUNS:
    path = run_path(**run)
    if run['method'] == 'es':
        if is_es_trained(path, **es_config(**run)):
            print(f'\n=== skip {run["label"]}: already trained at {path} ===')
            continue
        print(f'\n=== training {run["label"]} ({run["generations"]:,} generations) ===')
        run_es(**es_config(**run))
    else:
        agent, environment, name, trainer = run_config(**run)
        if is_trained(path, agent, environment, trainer):
            print(f'\n=== skip {name}: already trained at {path} ===')
            continue
        print(f'\n=== training {name} ({run["steps"]:,} steps) ===')
        train('import tonic.torch', agent, environment, name=name, trainer=trainer)

print(f'\n=== total training time: {time.time() - start:.0f}s ===')

## Learning curves

Mean test-episode reward vs cumulative environment steps, from each run's `log.csv`.

> The three curves share an axis but **do not count the same thing**: the tonic RL runs
> log test-episode length, while ES logs true environment interactions (a whole population
> per generation, so its sample cost is large and honest). Treat this as three separate
> "did it climb?" traces, not a head-to-head sample-efficiency race — the success bars
> below are the trustworthy comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
plot_performance([run_path(**run) for run in RUNS], ax=ax,
                 title='MLP-only foraging: PPO vs DDPG vs ES')
plt.tight_layout(); plt.show()

## Physics-only success — the money plot

The fraction of `EVAL_EPISODES` fresh episodes whose head reaches the food. Aggregate
training reward is misleading in this env (shaped, and different scales per method), so
this is the metric to trust. An **untrained MLP** anchors the floor.

In [ ]:
rates = {'untrained MLP': success_rate(*untrained_mlp_policy()[:2])}
for run in RUNS:
    pol, env, _ = load_policy(run)
    rates[run['label']] = success_rate(pol, env)

for k, v in rates.items():
    print(f'{k:22}: {v * 100:.0f}%')

colors = ['#bbbbbb', '#264653', '#e76f51', '#2a9d8f'][:len(rates)]
bars = plt.bar(range(len(rates)), [v * 100 for v in rates.values()], color=colors, ec='k')
plt.xticks(range(len(rates)), list(rates), rotation=15, ha='right')
plt.bar_label(bars, fmt='%.0f%%'); plt.ylabel('success (%)'); plt.ylim(0, 100)
plt.title(f'reaching food, {EVAL_EPISODES} episodes (eval seed {EVAL_SEED})')
plt.tight_layout(); plt.show()

## Watch the best baseline forage

Renders one episode of whichever run scored highest above (a fresh eval seed), with its
head path and the food it reached.

In [ ]:
best = max(RUNS, key=lambda r: rates[r['label']])
pol, env, phys = load_policy(best, seed=3)
ep = forage_episode(pol, env, phys, steps=1000)
print(f'{best["label"]}: food reached this episode: {len(ep["eats"])}')
show_video(ep['frames'][::2], f'output_videos/forage_{best["label"]}.mp4', fps=30)

In [ ]:
plot_path(ep, f'{best["label"]} (trained): head path and food')